In [172]:
from enum import Enum
from abc import ABC, abstractmethod
import copy


To do:

ich mache es ja so dass ich vin jeder figur die position als eigenschaft des klassenobjekts speichere. jetzt muss ich noch einbauen, dass die frühere figur verschwindet wenn sie geschlagen wurde.

v2

In [173]:
class Color(Enum):
    WHITE = "white"
    BLACK = "black"



class Player():
    def __init__(self, name, color):
        self.name = name
        self.color = color



class Board():
    def __init__(self):
        self.grid = [
        [None for _ in range(8)] for _ in range(8)
        ]
        self.white_king_position = (0, 4)
        self.black_king_postition = (7, 4)
        self.pieces_on_board = []
    
    
    def setup_pieces(self, color):
        
        row_pawns = 1 if color == Color.WHITE else 6
        row_others = 0 if color == Color.WHITE else 7

        for col in range (8):
            self.pieces_on_board.append(Pawn(color, position=(col, row_pawns),board=self))
            
        
        self.pieces_on_board.append(Rook(color, position=(0, row_others),board=self))
        self.pieces_on_board.append(Rook(color, position=(7, row_others),board=self))
        self.pieces_on_board.append(Knight(color, position=(1, row_others),board=self))
        self.pieces_on_board.append(Knight(color, position=(6, row_others),board=self))
        self.pieces_on_board.append(Bishop(color, position=(2, row_others),board=self))
        self.pieces_on_board.append(Bishop(color, position=(5, row_others),board=self))
        self.pieces_on_board.append(Queen(color, position=(3, row_others),board=self))
        self.pieces_on_board.append(King(color, position=(4, row_others),board=self))
            

    def reset_grid(self):
        self.grid = [[None for _ in range(8)] for _ in range(8)]

    
    def update_grid(self):
        self.reset_grid()
        for piece in self.pieces_on_board:
            x, y = piece._position
            self.grid[y][x] = piece


    def initialize(self):
        self.setup_pieces(Color.WHITE)
        self.setup_pieces(Color.BLACK)
        self.update_grid()
    
        
    def is_empty(self, square_coords):
        x, y = square_coords
        square = self.grid[y][x]
        return square == None
    

    def in_bounds(self, square_coords):
        x, y = square_coords
        return (0 <= x <= 7 and 0 <= y <= 7)
    

    def clone(self):
        return copy.deepcopy(self)
        
    
    def is_under_attack(self, color):
        return False



class Move():
    def __init__(self):
        self.captured_pieces = []
        self.move_history = []


        

class Piece(ABC):
    def __init__(self, color, position, board):
        self._color = color
        self._position = position
        self._has_moved = False
        self.board = board
    

    @abstractmethod
    def get_pseudo_legal_moves(self):
        pass
    
    
    def move(self, end_position):
        if self._move_is_legal(end_position):
            self._position = end_position
            self._has_moved = True


    def get_legal_moves(self): # noch nicht ganz fertig
        legal_moves = []
        for move in self.get_pseudo_legal_moves():
            x_old, y_old = self._position
            simulated_board = self.board.clone()
            simulated_board.grid[y_old][x_old]._position = move
            simulated_board.update_grid()

            if not simulated_board.is_under_attack(self._color):
                legal_moves.append(move)
        return legal_moves
    

    def _move_is_legal(self, end_position):
        return end_position in self.get_legal_moves()




class Pawn(Piece):
    def __init__(self, color, position, board):
        super().__init__(color, position, board)
        self._en_passant_vulnerability = False


    def get_pseudo_legal_moves(self):
        possible_moves = []

        direction = 1 if self._color == Color.WHITE else -1
        start_row = 1 if self._color == Color.WHITE else 6
        x, y = self._position

        # One step ahead
        if self.board.in_bounds((x, y+direction)):
            if self.board.is_empty((x, y+direction)):
                possible_moves.append((x, y+direction))
        
        # Two steps ahead
        if y == start_row and self.board.is_empty((x, y+2*direction)):
            possible_moves.append((x, y+2*direction))
        
        # Take piece
        if self.board.in_bounds((x+1, y+direction)):
            if not self.board.is_empty((x+1, y+direction)):
                possible_moves.append((x+1, y+direction))
        if self.board.in_bounds((x-1, y+direction)):
            if not self.board.is_empty((x-1, y+direction)):
                possible_moves.append((x-1, y+direction))
        
        # En passant
        if self.board.in_bounds((x+1, y+direction)):
            if self.board.grid[y+direction][x+1] == Pawn:
                if self.board.grid[y+direction][x+1]._en_passant_vulnerability:
                    possible_moves.append((x+1, y+direction))
        if self.board.in_bounds((x-1, y+direction)):
            if self.board.grid[y+direction][x-1] == Pawn:
                if self.board.grid[y+direction][x-1]._en_passant_vulnerability:
                    possible_moves.append((x-1, y+direction))

        return possible_moves
    
    
    def move(self, end_position):
        if self._move_is_legal(end_position):
            _, y_old = self._position
            _, y_new = end_position
            direction = 1 if self._color == Color.WHITE else -1
            self._en_passant_vulnerability = True if (y_new-y_old) == 2*direction else False
            self._position = end_position
            self._has_moved = True
        



class Rook(Piece):
    def get_pseudo_legal_moves(self, board):
        pass



class Knight(Piece):
    def get_pseudo_legal_moves(self, board):
        possible_moves = []



class Bishop(Piece):
    def get_pseudo_legal_moves(self, board):
        pass



class Queen(Piece):
    def get_pseudo_legal_moves(self, board):
        pass



class King(Piece):
    def get_pseudo_legal_moves(self, board):
        pass


In [174]:
board = Board()
board.initialize()
board2 = board.clone()
board.grid[1][0]._position = (1, 1)
print(board.grid[1][0]._position)
print(board.pieces_on_board[0]._position)
print(board2.grid[1][0]._position)

(1, 1)
(1, 1)
(0, 1)


In [175]:
board, board2

(<__main__.Board at 0x106e69550>, <__main__.Board at 0x106e53750>)

In [176]:
board = Board()
board.initialize()
board.grid[1][0].get_legal_moves()

[(0, 2), (0, 3)]

In [ ]:
board.grid[1][0].move((0, 2))
board.update_grid()
board.grid[2][0].move((0, 3))
board.update_grid()
print(board.grid)

[[<__main__.Rook object at 0x11407b950>, <__main__.Knight object at 0x114079750>, <__main__.Bishop object at 0x11407a950>, <__main__.Queen object at 0x107bdc3e0>, <__main__.King object at 0x107bdc510>, <__main__.Bishop object at 0x11407aa50>, <__main__.Knight object at 0x11407ad50>, <__main__.Rook object at 0x11407bc50>], [None, <__main__.Pawn object at 0x107864360>, <__main__.Pawn object at 0x107864a60>, <__main__.Pawn object at 0x107864050>, <__main__.Pawn object at 0x107865e80>, <__main__.Pawn object at 0x1078657f0>, <__main__.Pawn object at 0x107865fd0>, <__main__.Pawn object at 0x1078672a0>], [None, None, None, None, None, None, None, None], [<__main__.Pawn object at 0x1078673f0>, None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], [<__main__.Pawn object at 0x107864c20>, <__main__.Pawn object at 0x107864b40>, <__main__.Pawn object at 0x107864d70>, <__main__.Pawn object at 0x107864e50>, <__ma